Ноутбук 2

Импорт библиотек

In [15]:

!pip install scikit-learn pandas numpy -q
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

Загрузка данных

In [16]:
df = pd.read_csv('train_data_10k.csv', encoding='utf-8')
print(f"Загружено статей: {len(df)}")

with open('kaggle_mapping.json', 'r', encoding='utf-8') as f:
    kaggle_mapping = json.load(f)
kaggle_mapping = {int(k): v for k, v in kaggle_mapping.items()}
print(f"Категории: {kaggle_mapping}")

Загружено статей: 77481
Категории: {0: 'Общество/Россия', 1: 'Экономика', 2: 'Силовые структуры', 3: 'Бывший СССР', 4: 'Спорт', 5: 'Забота о себе', 6: 'Строительство', 7: 'Туризм/Путешествия', 8: 'Наука и техника'}


Разбиение данных

In [17]:
print("\nПодготовка данных")

X = df[['text']]
y = df['category_id']

for cat_id in sorted(y.unique()):
    count = len(y[y == cat_id])
    print(f"  {cat_id} ({kaggle_mapping[cat_id]}): {count}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nОбучающая выборка: {len(X_train)} статей")
print(f"Тестовая выборка: {len(X_test)} статей")


Подготовка данных
  0 (Общество/Россия): 9947
  1 (Экономика): 9565
  2 (Силовые структуры): 9673
  3 (Бывший СССР): 6214
  4 (Спорт): 9420
  5 (Забота о себе): 8803
  6 (Строительство): 8825
  7 (Туризм/Путешествия): 7604
  8 (Наука и техника): 7430

Обучающая выборка: 58110 статей
Тестовая выборка: 19371 статей


Векторизация

In [18]:
# CountVectorizer
vec = CountVectorizer()
vec.fit(X_train['text'])

X_train_bow = vec.transform(X_train['text'])
X_test_bow = vec.transform(X_test['text'])

print(f"Размер матрицы признаков: {X_train_bow.shape}")
# Нормализация
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train_bow)
X_test_scaled = scaler.transform(X_test_bow)


Размер матрицы признаков: (58110, 286806)


Логистическая регрессия

In [19]:
clf = LogisticRegression(max_iter=200, random_state=42)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[kaggle_mapping[i] for i in sorted(kaggle_mapping.keys())]))

Accuracy: 0.8234 (82.34%)

Classification Report:
                    precision    recall  f1-score   support

   Общество/Россия       0.65      0.67      0.66      2487
         Экономика       0.85      0.86      0.85      2391
 Силовые структуры       0.90      0.93      0.91      2418
       Бывший СССР       0.68      0.74      0.71      1554
             Спорт       0.91      0.86      0.89      2355
     Забота о себе       0.86      0.85      0.86      2201
     Строительство       0.89      0.87      0.88      2206
Туризм/Путешествия       0.86      0.84      0.85      1901
   Наука и техника       0.81      0.77      0.79      1858

          accuracy                           0.82     19371
         macro avg       0.82      0.82      0.82     19371
      weighted avg       0.83      0.82      0.82     19371



для kaggle

In [20]:
test_df = pd.read_csv('test_news.csv')
print(f"Тестовых новостей: {len(test_df)}")

X_test_kaggle = vec.transform(test_df['content'])
X_test_kaggle_scaled = scaler.transform(X_test_kaggle)
predictions = clf.predict(X_test_kaggle_scaled)

submission = pd.read_csv('base_submission_news.csv')
submission['topic'] = predictions
submission.to_csv('for_kaggle.csv', index=False)

print("\nРаспределение предсказаний:")
# Счет с 1 для удобства чтения
for i, cat_id in enumerate(sorted(np.unique(predictions)), 1):
    count = np.sum(predictions == cat_id)
    print(f"  {i}. Категория {cat_id} ({kaggle_mapping[cat_id]}): {count} новостей")


Тестовых новостей: 26275

Распределение предсказаний:
  1. Категория 0 (Общество/Россия): 10446 новостей
  2. Категория 1 (Экономика): 1669 новостей
  3. Категория 2 (Силовые структуры): 2708 новостей
  4. Категория 3 (Бывший СССР): 2013 новостей
  5. Категория 4 (Спорт): 2494 новостей
  6. Категория 5 (Забота о себе): 1912 новостей
  7. Категория 6 (Строительство): 1549 новостей
  8. Категория 7 (Туризм/Путешествия): 1413 новостей
  9. Категория 8 (Наука и техника): 2071 новостей


Скачиваем готовый файл

In [21]:
from google.colab import files
files.download('for_kaggle.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>